# lagKAN Tutorial
This notebook gives an example on how to use lagKAN. The dyn-BFC-2000-1 dataset in ../data/ was taken from the BEELINE benchmark (https://zenodo.org/records/19009603).

In [1]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import lagkan

DATA_DIR = Path("../data/dyn-BFC-2000-1")

## 1. Load Data
We load the raw CSV files from the data folder. 

ExpressionData.csv contains the genes as rows and cells as columns. Pseudotime.csv contains the cells as rows and lineages as columns. If only the raw counts matrix is available it is necessary to run a trajectory inference algorithm first to get the pseudotime values of each cell

In [2]:
# Read the expression and pseudotime matrix
expression_df = pd.read_csv(DATA_DIR / "ExpressionData.csv", header=0, index_col=0)
pt_df = pd.read_csv(DATA_DIR / "PseudoTime.csv", header=0, index_col=0)

# Ensure cells in both dataframes align
pt_df = pt_df.reindex(expression_df.columns)

print(f"Loaded Expression Matrix Shape: {expression_df.shape} (Genes x Cells)")
print(f"Loaded Pseudotime Matrix Shape: {pt_df.shape} (Cells x Lineages)")

Loaded Expression Matrix Shape: (10, 2000) (Genes x Cells)
Loaded Pseudotime Matrix Shape: (2000, 2) (Cells x Lineages)


## 2. Generate Matrix Inputs for the Inference Loop
The following matrices are necessary to for training the KAN:
- The raw expression counts of shape: (Cells x Genes)
- The pseudotime value of each cell of shape: (Cells x Lineages)
- A boolean mask corresponding to the lineage assignment of each cell of shape: (Cells x Lineages)

In [3]:
# Transpose the expressions so the format maps to (n_cells, n_genes)
raw_counts = expression_df.values.T             

# Replace na values in the pseudotime matrix with 0.0
pseudotime = pt_df.fillna(0.0).values           

# Construct the boolean mask by mapping na to False and any number to True
# This step should be adjusted, depending on the trajectory inference method
lineage_assignment = pt_df.notna().values       

gene_names = expression_df.index.values

print(f"Final Count Shape: {raw_counts.shape}")
print(f"Lineage Mask Shape: {lineage_assignment.shape}")
print(f"Number of genes to evaluate: {len(gene_names)}")

Final Count Shape: (2000, 10)
Lineage Mask Shape: (2000, 2)
Number of genes to evaluate: 10


## 3. Running  Inference
Before executing the inference algorithm it is also necessary to provide a relative time lag `dt`. Since the lag is relative it should be between 0 and 1. The absolute lag is then calculate for each lineage as `(max(pseudotime) - min(pseudotime)) * dt`. The default value of `dt` 0.08. The sparsity of the network can be increased via L1 Regularization by providing a higher `lamb_l1` value.

In [4]:
ranked_edges_df = lagkan.infer_grn(
    raw_counts=raw_counts,
    pseudotime=pseudotime,
    lineage_assignment=lineage_assignment,
    gene_names=gene_names,
    dt=0.08,
    epochs=100,     # Default is 400
    lr=0.01,
    lamb_l1=0.02    
)

[1/10] Processing gene: g1...
[2/10] Processing gene: g10...
[3/10] Processing gene: g2...
[4/10] Processing gene: g3...
[5/10] Processing gene: g4...
[6/10] Processing gene: g5...
[7/10] Processing gene: g6...
[8/10] Processing gene: g7...
[9/10] Processing gene: g8...
[10/10] Processing gene: g9...


## 4. Evaluate Network Output Structure
`infer_grn()` returns a list of edges between two genes and the weights (i.e. confidence) of the edge as a DataFrame sorted in descending order by (absolute) weight. Activators have a positive weight, repressors have a negative weight.

In [5]:
final_beeline_df = ranked_edges_df[["Gene1", "Gene2", "EdgeWeight"]]

print("Top 10 High-Confidence Inferred Links:")
print(final_beeline_df.head(10))

total_links = len(final_beeline_df)
activators = len(final_beeline_df[final_beeline_df["EdgeWeight"] > 0])
repressors = len(final_beeline_df[final_beeline_df["EdgeWeight"] < 0])

print(f"Total edges: {total_links}")
print(f"Activating edges: {activators}")
print(f"Repressing edges: {repressors}")

Top 10 High-Confidence Inferred Links:
  Gene1 Gene2  EdgeWeight
0    g4    g6    0.985789
1    g5    g7    0.878704
2    g8    g9    0.775251
3    g1    g3    0.724638
4    g1    g2    0.681441
5    g3    g4    0.622920
6    g3    g5    0.601986
7    g8    g4   -0.575985
8    g8    g5   -0.568573
9    g2    g3    0.545708
Total edges: 90
Activating edges: 62
Repressing edges: 28


## 5. AUROC and AUPRC Calculation
To evaluate how accurately lagKAN reconstructed the network, we will load the reference ground truth network and calculate the Area Under the Receiver Operating Characteristic (AUROC) and Area Under the Precision-Recall Curve (AUPRC).

In [6]:
from sklearn.metrics import roc_auc_score, average_precision_score

# Load the ground truth network
gt_df = pd.read_csv(DATA_DIR / "GroundTruthNetwork.csv", header=0)

# Create a set of true directed edges
true_edges = set(zip(gt_df["Gene1"], gt_df["Gene2"]))

# Map true labels to predictions
# If an edge exists in the ground truth, label is 1, otherwise 0
y_true = [
    1 if (row["Gene1"], row["Gene2"]) in true_edges else 0 
    for _, row in ranked_edges_df.iterrows()
]

# Use the absolute edge weights for calculating metrics
y_scores = abs(ranked_edges_df["EdgeWeight"].values)

# Calculate metrics
auroc = roc_auc_score(y_true, y_scores)
auprc = average_precision_score(y_true, y_scores)

# Calculate random baseline for AUPRC
total_possible_edges = len(ranked_edges_df)
total_true_edges_found = sum(y_true)
random_baseline = total_true_edges_found / total_possible_edges

print(f"Evaluation Metrics on dyn-BFC-2000-1:")
print(f"AUROC: {auroc:.4f}")
print(f"AUPRC: {auprc:.4f} (Random Baseline: {random_baseline:.4f})")
print(f"AUPRC Ratio: {auprc / random_baseline:.2f}x better than random")

Evaluation Metrics on dyn-BFC-2000-1:
AUROC: 0.8827
AUPRC: 0.7738 (Random Baseline: 0.1667)
AUPRC Ratio: 4.64x better than random
